In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt

In [3]:
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)

stk_data = yf.download("RELIANCE.NS", start=start, end=end)

[*********************100%***********************]  1 of 1 completed


In [4]:
stk_data

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2013-06-03,166.953644,170.651240,166.392105,170.301606,14165128
2013-06-04,165.322052,169.475242,165.099566,167.525789,14752690
2013-06-05,169.941391,170.428754,165.385594,165.385594,12748842
2013-06-06,167.822449,170.344022,166.996060,169.517633,17113393
2013-06-07,166.042511,169.941418,165.491580,168.288612,9420701
...,...,...,...,...,...
2022-02-04,1060.769531,1068.573059,1056.128431,1065.183162,11061241
2022-02-07,1054.308472,1072.372440,1048.802746,1065.638259,10714467


In [5]:
# The dataframe is not normal columns, it’s a MultiIndex (two-level columns).
# So, flattening the columns:

stk_data.columns = stk_data.columns.get_level_values(0)

In [6]:
stk_data=stk_data[["Open", "High", "Low", "Close"]]

In [7]:
stk_data

Price,Open,High,Low,Close
Date,,,,
2013-06-03,170.301606,170.651240,166.392105,166.953644
2013-06-04,167.525789,169.475242,165.099566,165.322052
2013-06-05,165.385594,170.428754,165.385594,169.941391
2013-06-06,169.517633,170.344022,166.996060,167.822449
2013-06-07,168.288612,169.941418,165.491580,166.042511
...,...,...,...,...
2022-02-04,1065.183162,1068.573059,1056.128431,1060.769531
2022-02-07,1065.638259,1072.372440,1048.802746,1054.308472
2022-02-08,1060.473750,1073.828376,1051.077695,1072.031006


In [8]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1 = Ms.fit_transform(stk_data)
print("Len:", data1.shape)

Len: (2144, 4)


In [9]:
data1 = pd.DataFrame(data1, columns=["Open", "High", "Low", "Close"])

### Train - Test Split

In [10]:
training_size = round(len(data1) * 0.80)
print(training_size)

X_train = data1[:training_size]
X_test = data1[training_size:]
print("X_train length:", X_train.shape)
print("X_test length:", X_test.shape)

y_train = data1[:training_size]
y_test = data1[training_size:]
print("y_train length:", y_train.shape)
print("y_test length:", y_test.shape)

1715
X_train length: (1715, 4)
X_test length: (429, 4)
y_train length: (1715, 4)
y_test length: (429, 4)


### Model Creation

#### VAR Model

In [11]:
performance = {"Model": [], "RMSE": [], "MaPe": [], "Lag": [], "Test": []}

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error


def combination_varmax(dataset, list1):

    print("Running for:", list1)  
    # Display which variables we are using

    datasetTwo = dataset[list1]  
    # Select only required columns (multivariate input)

    test_obs = 28  
    # Number of test observations (last 28 days)

    train = datasetTwo[:-test_obs]  
    # Training data (all except last 28)

    test = datasetTwo[-test_obs:]  
    # Testing data (last 28 for validation)

    best_aic = float("inf")  
    # Initialize best AIC as very large value

    best_order = None  
    # Store best (p,q) combination

    best_model = None  
    # Store best trained model

    # Try different combinations of (p,q)
    for p in range(1,4):  
        # AR terms (lags)

        for q in range(0,3):  
            # MA terms (error terms)

            try:
                model = VARMAX(train, order=(p,q))  
                # Create VARMAX model with (p,q)

                result = model.fit(disp=False)  
                # Fit model (disp=False → no verbose output)

                print("Order (p,q):", (p,q), "AIC:", result.aic)  
                # Print model order and its AIC

                if result.aic < best_aic:  
                    # Check if this model is better (lower AIC)

                    best_aic = result.aic  
                    # Update best AIC

                    best_order = (p,q)  
                    # Save best (p,q)

                    best_model = result  
                    # Save best model

            except:
                continue  
                # Skip combinations that fail

    print("Best Order Selected:", best_order)  
    # Show final selected (p,q)

    result = best_model  
    # Use best model

    pred = result.forecast(steps=test_obs)  
    # Forecast next 28 days

    # Evaluation
    mse = mean_squared_error(test, pred)  
    # Calculate Mean Squared Error

    rmse = round(np.sqrt(mse), 3)  
    # Convert to RMSE (actual scale)

    mape = mean_absolute_percentage_error(test, pred)  
    # Calculate percentage error

    # Store results
    performance["Model"].append(list1) 
    performance["RMSE"].append(rmse)  
    performance["MaPe"].append(mape)  
    performance["Lag"].append(best_order)  
    performance["Test"].append(test_obs)  
    
    perf = pd.DataFrame(performance)  

    return perf, result, pred  

In [ ]:
combination_varmax(data1, ["Close", "High"])
combination_varmax(data1, ["Close", "High", "Open"])
combination_varmax(data1, ["Close", "High", "Open", "Low"])

In [ ]:
perf = pd.DataFrame(performance)
perf